# IX-World — Hausa Ajami V9 — nouveau dataset

Split commun : **4972 train / 200 validation / 401 test reconstruit ; manuscrits disjoints**. Le nouveau test est isole par manuscrit. Ce notebook pilote le code versionne du depot IX-World ; il ne duplique pas son implementation. Aucune execution ni aucun resultat ne sont pre-remplis.

Choisir ACTION : `check`, `train` ou `test`. `train` effectue aussi le test final avec le meilleur checkpoint de validation. V13 reste une proposition, sans modele implemente.


**Campagne depuis zero du 16 octobre 2026 : initialisation aleatoire pour les quatre versions, aucun ancien checkpoint charge pour le train.**

Chaque epoque affiche le bilan detaille et cinq exemples de validation. Le bilan final donne CER, WER, UWER et taux exact pour train/validation/test, avec le meme checkpoint selectionne sur la validation. UWER : casse et diacritiques normalises, ponctuation conservee.

Les 5573 images valides sont utilisees ; seuls les deux cas d’annotation contradictoire restent en quarantaine. Le test public initial est reconstruit par manuscrit. Sources 1 et 2 regroupees par copiste connu. Identites des autres copistes non documentees. Aucun ancien checkpoint pour cette nouvelle campagne. Guide : README_HAUSA_GROUPED_20261016.md.


In [ ]:
from pathlib import Path
import os
import subprocess
import json

BASE = Path(r"D:\ai_test")
CODE = BASE / "OCR-AJAMI-Hausa"
DATASET = BASE / "Hausa_repo_2026-10-15"
PROTOCOL = BASE / "protocol_hausa_all_grouped_20261016"
VERSION = "v9"
RUN = BASE / "runs_hausa_all_grouped_20261016" / f"{VERSION}_seed42"
ACTION = "train"  # check / train / test ; train repart de zero
EPOCHS = 100

candidates = [os.environ.get("HAUSA_PYTHON"), str(BASE / ".venv-hausa/Scripts/python.exe"), r"C:\Users\admin\Documents\ChatGPT\AJAMI\.venv-v10\Scripts\python.exe"]
PYTHON = next((p for p in candidates if p and Path(p).is_file()), None)
if PYTHON is None:
    raise FileNotFoundError("Installer Python CUDA ou definir HAUSA_PYTHON.")
assert (CODE / "train_hausa.py").is_file(), CODE
assert (PROTOCOL / "split.json").is_file(), PROTOCOL
print("Python :", PYTHON)
print("Version :", VERSION, "Action :", ACTION, "Dataset :", DATASET)


In [ ]:
info = json.loads((PROTOCOL / "protocol.json").read_text(encoding="utf-8"))
print("Revision dataset :", info["dataset_revision"])
print("Split :", info["counts"])
print("SHA-256 split :", info["split_sha256"])
subprocess.run([PYTHON, "-c", "import torch; print(torch.__version__); print(torch.cuda.get_device_name(0)); assert torch.cuda.is_available()"], check=True)


In [ ]:
if ACTION not in {"check", "train", "test"}:
    raise ValueError(ACTION)
command = [PYTHON, "-X", "utf8", "-u", str(CODE / "train_hausa.py"), "--version", VERSION, "--require-group-disjoint", "--require-all-images", "--dataset", str(DATASET), "--protocol", str(PROTOCOL), "--output", str(RUN)]
if ACTION == "check":
    command += ["--check-only"]
elif ACTION == "test":
    command += ["--test-only"]
else:
    command += ["--fresh-start", "--epochs", str(EPOCHS)]
print(subprocess.list2cmdline(command))
subprocess.run(command, cwd=CODE, check=True)


In [ ]:
metrics = RUN / "metrics.json"
if metrics.exists():
    report = json.loads(metrics.read_text(encoding="utf-8"))
    print(f"Checkpoint selectionne : epoque {report['selected_epoch']}")
    print(f"{'Partition':12s} {'N':>6s} {'CER':>10s} {'WER':>10s} {'UWER':>10s} {'Exact':>10s}")
    for part, values in report["partitions"].items():
        print(f"{part:12s} {values['examples']:6d} {values['cer']:10.2%} {values['wer']:10.2%} {values['uwer']:10.2%} {values['exact_rate']:10.2%}")
    print("Train sans augmentation ; score sur donnees vues, sans valeur de generalisation.")
    print("Definition UWER :", report["uwer_definition"])
    print("CSV :", RUN / "metrics.csv")
else:
    print("Bilan complet absent. Apres la fin du train, choisir ACTION='test' et relancer la cellule de commande, sans reentrainement.")


In [ ]:
CREATE_PACKAGE = False
if CREATE_PACKAGE:
    destination = BASE / "packages" / f"hausa-{VERSION}-20261015"
    subprocess.run([PYTHON, str(CODE / "publish_hausa.py"), "package", "--run", str(RUN), "--destination", str(destination)], cwd=CODE, check=True)


## Commandes CMD individuelles

```cmd
cd /d D:\ai_test
lancer_v9.cmd
```

Recalculer le bilan train/validation/test sans reentrainer : `tester_v9.cmd`, apres la fin du train.

Le README detaille du modele sera produit a partir de ses resultats reels. Consulter `README_HAUSA_20261015.md` pour la publication locale HF ou via GitHub Release + Actions.

